In [8]:
!pip install langchain-experimental
!pip install langchain
!pip install langchain-openai
!pip install gradio-tools
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=850aa98d48104a6cd3eff90fb97d4d8b75f04f24d04c182d79ae4712712a4fbe
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [2]:
from langchain_community.agent_toolkits.load_tools import load_tools
from langgraph.prebuilt import create_react_agent
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import AzureOpenAI, ChatOpenAI
import os
import gradio as gr
from gradio_tools import (StableDiffusionTool, ImageCaptioningTool, StableDiffusionPromptGeneratorTool,
                          TextToVideoTool)
import pandas as pd
import numpy as np

In [3]:
with open("api_key.txt", "r") as file:
    api_key = file.read().strip()

In [4]:
df = pd.read_table("Restaurant_Reviews.tsv")

In [5]:
#os.environ["LLM_BASE_URL"] = "http://localhost:1234/v1"
os.environ["LLM_BASE_URL"] = ""
os.environ["LLM_MODEL"] = "gpt-4o-mini"
os.environ["OPENAI_API_KEY"] = api_key

In [6]:
LLM_BASE_URL = os.getenv("LLM_BASE_URL")
LLM_MODEL    = os.getenv("LLM_MODEL")
API_KEY      = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(
    base_url=LLM_BASE_URL,
    api_key=API_KEY,
    model=LLM_MODEL,
    temperature=0.2,
)


PREFIX = """
Eres un asistente para analizar un DataFrame llamado `df`.
Responde en español y da respuestas concretas.
Si necesitas calcular algo, usa Python internamente.
"""

agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    prefix=PREFIX,
    allow_dangerous_code=True,
    max_iterations=5,
    agent_executor_kwargs={"handle_parsing_errors": True},
)



def langchain_bot(prompt):
    if not prompt or not prompt.strip():
        return ""

    try:
        return agent.run(prompt)
    except Exception as e:
        return (
            "El agente no pudo completar la tarea.\n\n"
            "Prueba a reformular la pregunta de forma más concreta.\n\n"
            f"Detalle técnico:\n{e}"
        )



iface = gr.Interface(
    fn=langchain_bot,
    inputs=gr.Textbox(lines=5, placeholder="Escribe tu pregunta sobre el dataset..."),
    outputs=gr.Textbox(),
    title="LangChain Q&A (Pandas DataFrame Agent)",
    description="Responde preguntas sobre un DataFrame usando un agente de LangChain.",
    examples=[
        "¿Cuántas variables tiene el dataset?",
        "Lista las columnas y su tipo",
        "¿Qué columnas tienen valores nulos?",
        "Dame un resumen estadístico de las columnas numéricas",
    ],
)

iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7db023fdd57332567d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
tools = load_tools(["wikipedia"], llm=llm)

In [14]:
LLM_BASE_URL = os.getenv("LLM_BASE_URL")
LLM_MODEL    = os.getenv("LLM_MODEL")
API_KEY      = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(
    base_url=LLM_BASE_URL,
    api_key=API_KEY,
    model=LLM_MODEL,
    temperature=0.2,
)


PREFIX = """
Eres un asistente para responder preguntas de la tool recibida (wikipedia).
Responde en español y da respuestas concretas.
Si necesitas calcular algo, usa Python internamente.
"""

agent = create_pandas_dataframe_agent(
    llm,
    df,
    tools=tools,
    verbose=True,
    prefix=PREFIX,
    allow_dangerous_code=True,
    max_iterations=5,
    agent_executor_kwargs={"handle_parsing_errors": True},
)



def langchain_bot(prompt):
    if not prompt or not prompt.strip():
        return ""

    try:
        return agent.run(prompt)
    except Exception as e:
        return (
            "El agente no pudo completar la tarea.\n\n"
            "Prueba a reformular la pregunta de forma más concreta.\n\n"
            f"Detalle técnico:\n{e}"
        )



iface = gr.Interface(
    fn=langchain_bot,
    inputs=gr.Textbox(lines=5, placeholder="Escribe tu pregunta sobre el dataset..."),
    outputs=gr.Textbox(),
    title="LangChain Q&A (Wikipedia)",
    description="Responde preguntas haciendo uso de wikipedia.",
    examples=[
        "¿Cúal es el mejor equipo de fútbol del mundo?",
        "¿Qué es python?",
        "¿Qué es un modelo supervisado?",
    ],
)

iface.launch()


/usr/local/lib/python3.12/dist-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'tools': [WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/usr/local/lib/python3.12/dist-packages/wikipedia/__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000))]} which are no longer supported.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f9390e38a04ebb8c59.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
